# 🐯 Damru 14B Brain — Kaggle QLoRA → GGUF

**GPU:** T4 (or T4×2 — Unsloth uses one) · **Internet:** ON · add a Kaggle **Secret** named `HF_TOKEN`.

Trains the 14B primary brain with **Unsloth** on the clean, decontaminated `Damaru-ai/damru-train` split (built by `prep_training_data.py`), with completion-only loss, then exports **GGUF** (`q4_k_m` + `q5_k_m`) to `Damaru-ai/damru-14b-gguf` so the Space can serve Damru's own brain.

**Flow:** 1) install → 2) token + clone → 3) (optional) build split → 4) train + export → 5) flip Space to `OWN_MODEL_PRIMARY=1`.

In [ ]:
%%capture
# Unsloth + QLoRA stack for Kaggle T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.9" "transformers>=4.44" peft accelerate bitsandbytes datasets sentencepiece protobuf

In [ ]:
import os, sys
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secret')
except Exception as e:
    os.environ.setdefault('HF_TOKEN', '')  # or paste your token here
    print('No Kaggle secret -> add HF_TOKEN.', str(e)[:60])

!git clone -q https://github.com/Damru-AI/Damru-AI.git 2>/dev/null || echo 'repo already cloned'
sys.path.insert(0, 'Damru-AI/phase4')
print('phase4 on path')

In [ ]:
# OPTIONAL -- run ONLY if Damaru-ai/damru-train does NOT exist yet.
# Builds the clean, decontaminated, balanced split from damru-knowledge (streaming, CPU-friendly).
# import runpy; runpy.run_path('Damru-AI/phase4/prep_training_data.py', run_name='__main__')
print('skip unless damru-train missing')

In [ ]:
import os, importlib
os.environ['BASE_MODEL']  = 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
os.environ['TRAIN_REPO']  = 'Damaru-ai/damru-train'
os.environ['GGUF_REPO']   = 'Damaru-ai/damru-14b-gguf'
os.environ['LORA_REPO']   = 'Damaru-ai/damru-14b-lora'
os.environ['MAX_SEQ']     = '2048'
os.environ['BATCH']       = '2'
os.environ['GRAD_ACCUM']  = '8'
os.environ['EPOCHS']      = '1'
os.environ['QUANTS']      = 'q4_k_m,q5_k_m'
# os.environ['MAX_TRAIN_ROWS'] = '200000'   # uncomment for a fast first pass

import finetune_damru_14b as ft
importlib.reload(ft)
ft.main()

## ✅ After training

GGUF pushed to **`Damaru-ai/damru-14b-gguf`** (`q4_k_m` + `q5_k_m`); LoRA to `Damaru-ai/damru-14b-lora`.

**Make Damru use its own brain:** on the HF Space set **`OWN_MODEL_PRIMARY=1`** and point the GGUF loader at `Damaru-ai/damru-14b-gguf`, then restart. (Exact loader env wiring = Step 4b, after I read `open_brain.py`.)

**Session died after training?** Don't retrain — run `phase4/export_gguf.py` with `ADAPTER=Damaru-ai/damru-14b-lora` to export GGUF from the saved adapter.